In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import cupy as cp
cp.show_config()



OS                           : Linux-6.6.56+-x86_64-with-glibc2.35
Python Version               : 3.11.11
CuPy Version                 : 13.4.1
CuPy Platform                : NVIDIA CUDA
NumPy Version                : 1.26.4
SciPy Version                : 1.15.2
Cython Build Version         : 3.0.12
Cython Runtime Version       : 3.0.12
CUDA Root                    : /usr/local/cuda
nvcc PATH                    : /usr/local/cuda/bin/nvcc
CUDA Build Version           : 12080
CUDA Driver Version          : 12060
CUDA Runtime Version         : 12080 (linked to CuPy) / 12050 (locally installed)
CUDA Extra Include Dirs      : []
cuBLAS Version               : (available)
cuFFT Version                : 11203
cuRAND Version               : 10306
cuSOLVER Version             : (11, 6, 3)
cuSPARSE Version             : (available)
NVRTC Version                : (12, 5)
Thrust Version               : 200800
CUB Build Version            : 200800
Jitify Build Version         : <unknown>
cuDNN Buil

In [8]:
import os
from PIL import Image
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from cupyx.scipy.ndimage import convolve
import time
import pandas as pd

# Input & output directories
input_dir = "/kaggle/input/misc-images"
output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)

# Create log file
log_path = os.path.join(output_dir, "log.txt")
log_file = open(log_path, "w")
log_file.write("Image Processing Log\n=====================\n")

# Timing storage
timing_data = []

# Blur kernel for blue
kernel = cp.ones((3, 3)) / 9.0

# Track how many samples we saved R/G/B channel visuals for
saved_channels = 0
max_channel_saves = 3

# Process each image
for fname in os.listdir(input_dir):
    if fname.endswith(".tiff") or fname.endswith(".tif"):
        try:
            path = os.path.join(input_dir, fname)
            img = Image.open(path).convert("RGB")
            img_np = np.array(img, dtype=np.float32)
            cp_img = cp.array(img_np)

            start = time.time()

            # Split channels
            r, g, b = cp_img[:, :, 0], cp_img[:, :, 1], cp_img[:, :, 2]

            # Red: brighten
            r_mod = cp.clip(r * 1.2, 0, 255)

            # Green: contrast stretch
            g_mod = cp.clip((g - g.mean()) * 1.5 + g.mean(), 0, 255)

            # Blue: blur
            b_mod = convolve(b, kernel, mode='reflect')

            # Merge all
            merged = cp.stack([r_mod, g_mod, b_mod], axis=2).astype(cp.uint8)
            merged_np = cp.asnumpy(merged)

            # Save enhanced image
            base_name = fname.replace(".tiff", "").replace(".tif", "")
            out_path = os.path.join(output_dir, f"{base_name}_enhanced.png")
            Image.fromarray(merged_np).save(out_path)

            # Save R/G/B modified channels for first few images
            if saved_channels < max_channel_saves:
                Image.fromarray(cp.asnumpy(r_mod.astype(cp.uint8))).save(
                    os.path.join(output_dir, f"{base_name}_red_brightened.png"))
                Image.fromarray(cp.asnumpy(g_mod.astype(cp.uint8))).save(
                    os.path.join(output_dir, f"{base_name}_green_contrast.png"))
                Image.fromarray(cp.asnumpy(b_mod.astype(cp.uint8))).save(
                    os.path.join(output_dir, f"{base_name}_blue_blurred.png"))
                saved_channels += 1

            end = time.time()
            elapsed = round((end - start) * 1000, 2)

            log_file.write(f" {fname} processed in {elapsed} ms\n")
            print(f"{fname} processed in {elapsed} ms")

            timing_data.append({"Filename": fname, "Time (ms)": elapsed})

        except Exception as e:
            log_file.write(f" Failed on {fname}: {e}\n")
            print(f" Failed on {fname}: {e}")

# Save timing data
df = pd.DataFrame(timing_data)
df.to_csv(os.path.join(output_dir, "new_timing.csv"), index=False)

# Close log
log_file.write("\nAll images processed successfully.\n")
log_file.close()


7.1.10.tiff processed in 212.19 ms
4.2.06.tiff processed in 162.35 ms
5.1.13.tiff processed in 18.58 ms
7.1.06.tiff processed in 92.51 ms
4.1.01.tiff processed in 26.11 ms
4.1.02.tiff processed in 24.88 ms
5.1.12.tiff processed in 22.9 ms
4.1.07.tiff processed in 27.4 ms
4.2.07.tiff processed in 84.09 ms
7.1.05.tiff processed in 89.91 ms
5.2.09.tiff processed in 89.63 ms
7.1.07.tiff processed in 90.61 ms
7.2.01.tiff processed in 466.59 ms
5.1.10.tiff processed in 18.95 ms
7.1.09.tiff processed in 96.94 ms
ruler.512.tiff processed in 13.29 ms
4.2.01.tiff processed in 136.69 ms
7.1.08.tiff processed in 118.0 ms
7.1.02.tiff processed in 99.54 ms
5.2.08.tiff processed in 117.21 ms
4.1.03.tiff processed in 37.51 ms
house.tiff processed in 91.09 ms
4.1.05.tiff processed in 26.61 ms
gray21.512.tiff processed in 8.41 ms
4.2.03.tiff processed in 63.64 ms
5.1.09.tiff processed in 25.35 ms
4.1.04.tiff processed in 31.54 ms
5.3.02.tiff processed in 361.03 ms
4.2.05.tiff processed in 115.97 ms
4.1.

In [9]:
import shutil

# Zip the outputs directory
shutil.make_archive("sipi_outputs", 'zip', "/kaggle/working/outputs")


'/kaggle/working/sipi_outputs.zip'